# 1.2 Demo - Multiplex Streaming SDP with Delta Sinks and Iceberg Reads
**Lesson:** Introduction to Multiplex Streaming, Delta Sinks and Iceberg Reads

A message bus such as Kafka delivers **many topics through one stream**. The *Multiplex* pattern lands everything in **one Bronze table** and then fans out to one Silver table per topic, each with its own schema.

In this demo you will:
1. Land a multiplexed feed (`orders`, `customers`, `clickstream`) in `bronze_multiplex`, clustered by `topic`
2. Generate one Silver table per topic **in a loop** (pipeline metaprogramming) and quarantine unparseable payloads
3. Stream records **out** of the pipeline into an external **Delta sink** (`dp.create_sink` + `@dp.append_flow`)
4. Enable **UniForm** so the Silver tables also publish **Iceberg metadata**, then read them with an Iceberg client (PyIceberg) that knows nothing about Delta

## A. Classroom setup

In [ ]:
from adelab.classroom import setup
from adelab import datagen

DA = setup("1.2", reset=True)

MULTIPLEX_SOURCE = f"{DA.paths.landing}/multiplex"
SINK_PATH = f"{DA.paths.working_dir}/exports/clickstream_checkout"
for _ in range(2):
    datagen.land_multiplex(MULTIPLEX_SOURCE)

## B. The raw multiplexed feed
Each record carries `topic`, `key`, `partition`, `offset`, `timestamp`, and a `value` that holds a JSON payload whose shape **depends on the topic**.

In [ ]:
from pyspark.sql import functions as F

raw = spark.read.json(MULTIPLEX_SOURCE)
display(raw.groupBy("topic").count())
display(raw.select("topic", "key", "value").limit(6))

## C. The pipeline
- **`01_bronze_multiplex.py`**: one streaming table for all topics, `cluster_by=["topic"]` so each fan-out reads only its topic's files.
- **`02_silver_topics.py`**: a `for` loop calls a function that defines one `@dp.table` per topic. Each table parses `value` with `from_json`, has a `valid_payload` expectation, and sets the **UniForm** table properties. A quarantine table keeps unparseable payloads.
- **`03_sinks_and_gold.py`**: `dp.create_sink(..., format="delta")` plus an append flow that exports checkout clicks, and a Gold materialized view.

![Multiplex fan-out: multiplex JSON files land in bronze_multiplex clustered by topic; a Python loop generates silver_orders, silver_customers and silver_clickstream; unparseable payloads go to bronze_multiplex_quarantine; an append flow exports /cart and /checkout clicks to the clickstream_export Delta sink; gold_orders_by_region joins orders and customers](../images/1.2-multiplex-fanout.svg)

**How to read it:**
- The dashed group is the `for` loop in `02_silver_topics.py`: one call to `define_topic_table` per entry in `TOPICS`.
- The sink (dashed blue) is outside the pipeline. Only the append flow `export_checkout_clicks` can write to it.
- Section D below counts rows per box; section E reads the sink path.

In [ ]:
DA.show_source("pipelines/multiplex")

In [ ]:
pipe = DA.pipeline(
    "multiplex",
    source="pipelines/multiplex",
    configuration={"multiplex_source": MULTIPLEX_SOURCE, "clickstream_sink_path": SINK_PATH},
)
pipe.validate()

In [ ]:
update = pipe.run()

## D. Fan-out results

In [ ]:
%%sql
SHOW TABLES

In [ ]:
%%sql
SELECT 'bronze' AS layer, topic, count(*) AS rows FROM bronze_multiplex GROUP BY topic
UNION ALL SELECT 'silver_orders', 'orders', count(*) FROM silver_orders
UNION ALL SELECT 'silver_customers', 'customers', count(*) FROM silver_customers
UNION ALL SELECT 'silver_clickstream', 'clickstream', count(*) FROM silver_clickstream
UNION ALL SELECT 'quarantine', topic, count(*) FROM bronze_multiplex_quarantine GROUP BY topic
ORDER BY topic, layer

In [ ]:
%%sql
SELECT topic, key, value FROM bronze_multiplex_quarantine LIMIT 5

In [ ]:
display(spark.table("silver_orders").limit(5))

## E. Delta sink: streaming out of the pipeline
A sink is not a pipeline table. It is an **external** destination (a Delta path or table, or Kafka / Event Hubs on Databricks). Its append flow has its own checkpoint, so it delivers each record exactly once.

Sink rules to remember:
- Only **append flows** can write to a sink (no `@dp.table`, no AUTO CDC).
- Sinks don't support **expectations**. Validate the data upstream, as `silver_clickstream` does.
- A **full refresh** does not clean up a sink: its flows restart from the beginning and append all records **again**. Consumers of a sink must tolerate duplicates after a full refresh.

Expected below: only `/cart` and `/checkout` pages, and as many rows as those pages have in `silver_clickstream`.

In [ ]:
exported = spark.read.format("delta").load(SINK_PATH)
display(exported.groupBy("page").count())
print(f"{exported.count()} rows exported to {SINK_PATH}")

## F. UniForm: the same table, readable as Iceberg
The UniForm properties (`delta.enableIcebergCompatV2`, `delta.universalFormat.enabledFormats = 'iceberg'`) make every Delta commit also write **Iceberg metadata** next to the data. Column mapping is turned on automatically.

![UniForm: the silver_orders location holds one set of Parquet files plus a _delta_log folder and an Iceberg metadata folder; Spark reads the table as Delta and an Iceberg engine reads the same files as Iceberg](../images/1.2-uniform-iceberg.svg)

**How to read it:** `SHOW TBLPROPERTIES` shows the properties at the bottom of the picture, `iceberg.metadata_files` lists the right-hand `metadata/` folder, and PyIceberg plays the "Iceberg engine" box. Both readers return the same rows because they point to the same Parquet files.

In [ ]:
%%sql
SHOW TBLPROPERTIES silver_orders

In [ ]:
from adelab import iceberg

for f in iceberg.metadata_files(spark, "silver_orders"):
    print(f)

Now read the table as **Iceberg** with PyIceberg, a pure-Python Iceberg client with no Spark and no Delta. This is how Snowflake, Trino, BigQuery, or DuckDB see the table. On Databricks they connect through the **Unity Catalog Iceberg REST catalog** instead of a metadata file path.

In [ ]:
ice = iceberg.load_table(spark, "silver_orders")
print(ice.schema())
pdf = ice.scan().to_pandas()
print(f"Iceberg rows: {len(pdf)}   Delta rows: {spark.table('silver_orders').count()}")
pdf.head()

## G. New data arrives
Land another file and run again. The Iceberg view updates with the Delta table.

In [ ]:
datagen.land_multiplex(MULTIPLEX_SOURCE)
update = pipe.run()
print("Iceberg rows now:", len(iceberg.read_as_pandas(spark, "silver_orders")),
      "| Delta rows:", spark.table("silver_orders").count())

In [ ]:
%%sql
SELECT * FROM gold_orders_by_region ORDER BY revenue DESC

## H. Key takeaways
- **Multiplex**: one Bronze table per source system; cluster it by `topic` so each fan-out flow scans only its own topic.
- **Metaprogramming**: Python loops generate many similar tables from one definition. Use a function so each table captures its own `topic` value.
- **Sinks** push data to external systems from inside the pipeline, with exactly-once delivery through the flow checkpoint. Append flows only, no expectations, and a full refresh re-appends instead of truncating.
- **UniForm** gives Iceberg readers access without copying data. Requirements: deletion vectors off, column mapping on (set automatically).

In [ ]:
DA.cleanup()